In [1]:
import pandas as pd
import numpy as np
pd.options.display.max_columns = None
from openslide import open_slide
import glob

In [2]:
def filter_files(x, slides):
    if slides == 'ffpe':
        if '-DX' in x or 'rna_seq' in x:
            return True
        else:
            return False
    elif slides == 'ff':
        if '-TS' in x or 'rna_seq' in x:
            return True
        else:
            return False
            
    return None


def has_magnif(image_path):
    img = open_slide(image_path)
    magnification = img.properties.get("aperio.AppMag", None)
    if magnification in ["40", "20"]:
        return True
    else:
        return False

def get_slide_orientation(x):
    if '-TS' in x:
        return 'TS'
    elif '-BS' in x:
        return 'BS'
    elif 'rna_seq' in x:
        return 'RNA'
    elif '-DX' in x:
        return 'DX'
    else:
        return np.nan 

In [3]:
slide_type = 'ff' # ff
sample_sheet = 'gdc_sample_sheet.2025-04-11'

In [4]:
df = pd.read_csv(f'data/{sample_sheet}.tsv', sep='\t')
df = df[df['File Name'].apply(lambda x: filter_files(x, slides=slide_type))]
df

,File ID,File Name,Data Category,Data Type,Project ID,Case ID,Sample ID,Tissue Type,Tumor Descriptor,Specimen Type,Preservation Method
2,c6607558-a740-46b3-8f35-67cdb467e0f7,TCGA-60-2712-01A-01-TS1.c5829f1c-421a-4498-b8c...,Biospecimen,Slide Image,TCGA-LUSC,TCGA-60-2712,TCGA-60-2712-01A,Tumor,Primary,Unknown,Unknown
5,044d098e-c65f-4152-8939-3c315aed0073,80ca4c12-21e8-45d1-8820-537b99bce32d.rna_seq.a...,Transcriptome Profiling,Gene Expression Quantification,TCGA-LUSC,TCGA-60-2712,TCGA-60-2712-01A,Tumor,Primary,Unknown,Unknown
7,f95d3d81-2433-4018-a042-f24b9f04946a,545f9937-b128-4f44-8b12-ded0fb79bf3f.rna_seq.a...,Transcriptome Profiling,Gene Expression Quantification,TCGA-LUSC,TCGA-56-7221,TCGA-56-7221-01A,Tumor,Primary,Solid Tissue,Unknown
10,0535a463-fc37-4c38-b88f-0eee6042ba55,TCGA-56-7221-01A-01-TS1.97c4d16d-5160-46ac-8f4...,Biospecimen,Slide Image,TCGA-LUSC,TCGA-56-7221,TCGA-56-7221-01A,Tumor,Primary,Solid Tissue,Unknown
12,da8679e6-96fc-4874-83c8-a3e3b0fe20d4,9b86812f-b1ee-4b6d-9691-8587f2487c4a.rna_seq.a...,Transcriptome Profiling,Gene Expression Quantification,TCGA-LUSC,TCGA-21-A5DI,TCGA-21-A5DI-01A,Tumor,Primary,Solid Tissue,OCT
...,...,...,...,...,...,...,...,...,...,...,...
2740,88ae3134-5c53-4595-95df-0990893736d5,TCGA-NC-A5HE-01A-01-TSA.A1F9C4DD-445C-45E4-A2A...,Biospecimen,Slide Image,TCGA-LUSC,TCGA-NC-A5HE,TCGA-NC-A5HE-01A,Tumor,Primary,Solid Tissue,OCT
2745,c8a68cbe-87a1-44b3-bf4b-68ba0de91994,3c174eac-9153-4c66-8da4-969925c2c4c4.rna_seq.a...,Transcriptome Profiling,Gene Expression Quantification,TCGA-LUSC,TCGA-66-2783,TCGA-66-2783-01A,Tumor,Primary,Unknown,Unknown
2746,8d111769-bfcd-4d44-a5e2-81dc7ca0eff5,TCGA-L3-A4E7-01A-01-TSA.77623172-E179-41DE-969...,Biospecimen,Slide Image,TCGA-LUSC,TCGA-L3-A4E7,TCGA-L3-A4E7-01A,Tumor,Primary,Solid Tissue,OCT
2747,5c3ec03d-d9eb-48e8-b132-5733dcf83131,5332a617-f178-4197-8ed4-2f49bf8751e0.rna_seq.a...,Transcriptome Profiling,Gene Expression Quantification,TCGA-LUSC,TCGA-66-2795,TCGA-66-2795-01A,Tumor,Primary,Solid Tissue,Unknown


In [5]:
if slide_type == 'ffpe':
    paired_samples = df[['Case ID', 'Data Type']].value_counts().reset_index().value_counts('Case ID').reset_index()
elif slide_type == 'ff':
    paired_samples = df[['Sample ID', 'Data Type']].value_counts().reset_index().value_counts('Sample ID').reset_index()
paired_samples = paired_samples[paired_samples['count'] > 1]
paired_samples

,Sample ID,count
0,TCGA-XC-AA0X-01A,2
1,TCGA-18-3406-01A,2
2,TCGA-98-A53A-01A,2
3,TCGA-98-A53B-01A,2
4,TCGA-98-A53C-01A,2
...,...,...
493,TCGA-18-3415-01A,2
494,TCGA-18-3407-01A,2
495,TCGA-18-3408-01A,2
496,TCGA-18-3409-01A,2


In [6]:
if slide_type == 'ffpe':
    df = df[df['Case ID'].isin(paired_samples['Case ID'].values)]
elif slide_type == 'ff':
    df = df[df['Sample ID'].isin(paired_samples['Sample ID'].values)]
df

,File ID,File Name,Data Category,Data Type,Project ID,Case ID,Sample ID,Tissue Type,Tumor Descriptor,Specimen Type,Preservation Method
2,c6607558-a740-46b3-8f35-67cdb467e0f7,TCGA-60-2712-01A-01-TS1.c5829f1c-421a-4498-b8c...,Biospecimen,Slide Image,TCGA-LUSC,TCGA-60-2712,TCGA-60-2712-01A,Tumor,Primary,Unknown,Unknown
5,044d098e-c65f-4152-8939-3c315aed0073,80ca4c12-21e8-45d1-8820-537b99bce32d.rna_seq.a...,Transcriptome Profiling,Gene Expression Quantification,TCGA-LUSC,TCGA-60-2712,TCGA-60-2712-01A,Tumor,Primary,Unknown,Unknown
7,f95d3d81-2433-4018-a042-f24b9f04946a,545f9937-b128-4f44-8b12-ded0fb79bf3f.rna_seq.a...,Transcriptome Profiling,Gene Expression Quantification,TCGA-LUSC,TCGA-56-7221,TCGA-56-7221-01A,Tumor,Primary,Solid Tissue,Unknown
10,0535a463-fc37-4c38-b88f-0eee6042ba55,TCGA-56-7221-01A-01-TS1.97c4d16d-5160-46ac-8f4...,Biospecimen,Slide Image,TCGA-LUSC,TCGA-56-7221,TCGA-56-7221-01A,Tumor,Primary,Solid Tissue,Unknown
12,da8679e6-96fc-4874-83c8-a3e3b0fe20d4,9b86812f-b1ee-4b6d-9691-8587f2487c4a.rna_seq.a...,Transcriptome Profiling,Gene Expression Quantification,TCGA-LUSC,TCGA-21-A5DI,TCGA-21-A5DI-01A,Tumor,Primary,Solid Tissue,OCT
...,...,...,...,...,...,...,...,...,...,...,...
2739,5bb8ce26-770f-48cc-a481-466469d34073,4f179a2f-1e37-4ad9-9f21-0a6f1ecdf03e.rna_seq.a...,Transcriptome Profiling,Gene Expression Quantification,TCGA-LUSC,TCGA-NC-A5HE,TCGA-NC-A5HE-01A,Tumor,Primary,Solid Tissue,OCT
2740,88ae3134-5c53-4595-95df-0990893736d5,TCGA-NC-A5HE-01A-01-TSA.A1F9C4DD-445C-45E4-A2A...,Biospecimen,Slide Image,TCGA-LUSC,TCGA-NC-A5HE,TCGA-NC-A5HE-01A,Tumor,Primary,Solid Tissue,OCT
2745,c8a68cbe-87a1-44b3-bf4b-68ba0de91994,3c174eac-9153-4c66-8da4-969925c2c4c4.rna_seq.a...,Transcriptome Profiling,Gene Expression Quantification,TCGA-LUSC,TCGA-66-2783,TCGA-66-2783-01A,Tumor,Primary,Unknown,Unknown
2746,8d111769-bfcd-4d44-a5e2-81dc7ca0eff5,TCGA-L3-A4E7-01A-01-TSA.77623172-E179-41DE-969...,Biospecimen,Slide Image,TCGA-LUSC,TCGA-L3-A4E7,TCGA-L3-A4E7-01A,Tumor,Primary,Solid Tissue,OCT


In [7]:
metadata = []
if slide_type == 'ffpe':
    iterate_over = 'Case ID'
elif slide_type == 'ff':
    iterate_over = 'Sample ID'

for iterate_over_id in df[iterate_over].unique():
    tab = df[df[iterate_over] == iterate_over_id]

    tab_rna = tab[tab['Data Type'] == 'Gene Expression Quantification']
    tab_slide = tab[tab['Data Type'] == 'Slide Image']
    

    for _, row_rna in tab_rna.iterrows():
        for _, row_slide in tab_slide.iterrows():
            if row_rna['Tumor Descriptor'] == row_slide['Tumor Descriptor']:
                metadata.append([row_slide['File Name'], row_rna['File Name'], row_rna['Case ID'], 
                                 row_slide['Sample ID'], row_rna['Sample ID'], row_slide['Tumor Descriptor']])
                
metadata = pd.DataFrame(metadata, columns=['image_path', 'rna_path', 'case_id', 'sample_slide_id', 'sample_rna_id', 'sample_type'])
metadata

,image_path,rna_path,case_id,sample_slide_id,sample_rna_id,sample_type
0,TCGA-60-2712-01A-01-TS1.c5829f1c-421a-4498-b8c...,80ca4c12-21e8-45d1-8820-537b99bce32d.rna_seq.a...,TCGA-60-2712,TCGA-60-2712-01A,TCGA-60-2712-01A,Primary
1,TCGA-56-7221-01A-01-TS1.97c4d16d-5160-46ac-8f4...,545f9937-b128-4f44-8b12-ded0fb79bf3f.rna_seq.a...,TCGA-56-7221,TCGA-56-7221-01A,TCGA-56-7221-01A,Primary
2,TCGA-21-A5DI-01A-03-TS3.FD5286B2-AC59-425F-B94...,9b86812f-b1ee-4b6d-9691-8587f2487c4a.rna_seq.a...,TCGA-21-A5DI,TCGA-21-A5DI-01A,TCGA-21-A5DI-01A,Primary
3,TCGA-43-7657-11A-01-TS1.7de37250-e538-4040-881...,4c84a3bd-3b6f-4627-a4d9-acd3b9669608.rna_seq.a...,TCGA-43-7657,TCGA-43-7657-11A,TCGA-43-7657-11A,Not Applicable
4,TCGA-94-7033-01A-01-TS1.d38f20aa-3a65-4af3-a58...,23f1ad0c-c9d5-408f-bba8-1bb71364007b.rna_seq.a...,TCGA-94-7033,TCGA-94-7033-01A,TCGA-94-7033-01A,Primary
...,...,...,...,...,...,...
503,TCGA-43-6773-11A-01-TS1.6e328690-6ad9-4d61-a42...,187da62b-d4a8-4c9e-a97e-ae5cb3a2c658.rna_seq.a...,TCGA-43-6773,TCGA-43-6773-11A,TCGA-43-6773-11A,Not Applicable
504,TCGA-43-6143-01A-01-TS1.6716067a-f179-46cf-8d3...,3953b63a-1a63-4e97-b683-ae905b55e03e.rna_seq.a...,TCGA-43-6143,TCGA-43-6143-01A,TCGA-43-6143-01A,Primary
505,TCGA-43-A56V-01A-01-TSA.6B4B05A2-0CAC-4779-9C8...,50f7bf08-3f8b-492b-afdc-40b4f35060bf.rna_seq.a...,TCGA-43-A56V,TCGA-43-A56V-01A,TCGA-43-A56V-01A,Primary
506,TCGA-56-5897-01A-01-TS1.360c2b58-7187-4e91-a2e...,8175545d-b1a7-472e-aa7f-23adaece2ccd.rna_seq.a...,TCGA-56-5897,TCGA-56-5897-01A,TCGA-56-5897-01A,Primary


In [8]:
metadata['data_type_info'] = metadata['image_path'].apply(lambda x: get_slide_orientation(x))
metadata['data_type_info'].value_counts()

data_type_info
TS    508
Name: count, dtype: int64

In [9]:
metadata = metadata[metadata.apply(lambda x: len(glob.glob(f"data/*/*/{x.rna_path}")) > 0, axis=1)]
metadata = metadata[metadata.apply(lambda x: len(glob.glob(f"data/*/*/{x.image_path}")) > 0, axis=1)]
metadata

,image_path,rna_path,case_id,sample_slide_id,sample_rna_id,sample_type,data_type_info
0,TCGA-60-2712-01A-01-TS1.c5829f1c-421a-4498-b8c...,80ca4c12-21e8-45d1-8820-537b99bce32d.rna_seq.a...,TCGA-60-2712,TCGA-60-2712-01A,TCGA-60-2712-01A,Primary,TS
1,TCGA-56-7221-01A-01-TS1.97c4d16d-5160-46ac-8f4...,545f9937-b128-4f44-8b12-ded0fb79bf3f.rna_seq.a...,TCGA-56-7221,TCGA-56-7221-01A,TCGA-56-7221-01A,Primary,TS
2,TCGA-21-A5DI-01A-03-TS3.FD5286B2-AC59-425F-B94...,9b86812f-b1ee-4b6d-9691-8587f2487c4a.rna_seq.a...,TCGA-21-A5DI,TCGA-21-A5DI-01A,TCGA-21-A5DI-01A,Primary,TS
3,TCGA-43-7657-11A-01-TS1.7de37250-e538-4040-881...,4c84a3bd-3b6f-4627-a4d9-acd3b9669608.rna_seq.a...,TCGA-43-7657,TCGA-43-7657-11A,TCGA-43-7657-11A,Not Applicable,TS
4,TCGA-94-7033-01A-01-TS1.d38f20aa-3a65-4af3-a58...,23f1ad0c-c9d5-408f-bba8-1bb71364007b.rna_seq.a...,TCGA-94-7033,TCGA-94-7033-01A,TCGA-94-7033-01A,Primary,TS
...,...,...,...,...,...,...,...
503,TCGA-43-6773-11A-01-TS1.6e328690-6ad9-4d61-a42...,187da62b-d4a8-4c9e-a97e-ae5cb3a2c658.rna_seq.a...,TCGA-43-6773,TCGA-43-6773-11A,TCGA-43-6773-11A,Not Applicable,TS
504,TCGA-43-6143-01A-01-TS1.6716067a-f179-46cf-8d3...,3953b63a-1a63-4e97-b683-ae905b55e03e.rna_seq.a...,TCGA-43-6143,TCGA-43-6143-01A,TCGA-43-6143-01A,Primary,TS
505,TCGA-43-A56V-01A-01-TSA.6B4B05A2-0CAC-4779-9C8...,50f7bf08-3f8b-492b-afdc-40b4f35060bf.rna_seq.a...,TCGA-43-A56V,TCGA-43-A56V-01A,TCGA-43-A56V-01A,Primary,TS
506,TCGA-56-5897-01A-01-TS1.360c2b58-7187-4e91-a2e...,8175545d-b1a7-472e-aa7f-23adaece2ccd.rna_seq.a...,TCGA-56-5897,TCGA-56-5897-01A,TCGA-56-5897-01A,Primary,TS


In [10]:
is_magnif = metadata.apply(lambda x: has_magnif(glob.glob(f"data/*/*/{x.image_path}")[0]), axis=1)
(~is_magnif).sum()

0

In [11]:
metadata = metadata[is_magnif]
metadata

,image_path,rna_path,case_id,sample_slide_id,sample_rna_id,sample_type,data_type_info
0,TCGA-60-2712-01A-01-TS1.c5829f1c-421a-4498-b8c...,80ca4c12-21e8-45d1-8820-537b99bce32d.rna_seq.a...,TCGA-60-2712,TCGA-60-2712-01A,TCGA-60-2712-01A,Primary,TS
1,TCGA-56-7221-01A-01-TS1.97c4d16d-5160-46ac-8f4...,545f9937-b128-4f44-8b12-ded0fb79bf3f.rna_seq.a...,TCGA-56-7221,TCGA-56-7221-01A,TCGA-56-7221-01A,Primary,TS
2,TCGA-21-A5DI-01A-03-TS3.FD5286B2-AC59-425F-B94...,9b86812f-b1ee-4b6d-9691-8587f2487c4a.rna_seq.a...,TCGA-21-A5DI,TCGA-21-A5DI-01A,TCGA-21-A5DI-01A,Primary,TS
3,TCGA-43-7657-11A-01-TS1.7de37250-e538-4040-881...,4c84a3bd-3b6f-4627-a4d9-acd3b9669608.rna_seq.a...,TCGA-43-7657,TCGA-43-7657-11A,TCGA-43-7657-11A,Not Applicable,TS
4,TCGA-94-7033-01A-01-TS1.d38f20aa-3a65-4af3-a58...,23f1ad0c-c9d5-408f-bba8-1bb71364007b.rna_seq.a...,TCGA-94-7033,TCGA-94-7033-01A,TCGA-94-7033-01A,Primary,TS
...,...,...,...,...,...,...,...
503,TCGA-43-6773-11A-01-TS1.6e328690-6ad9-4d61-a42...,187da62b-d4a8-4c9e-a97e-ae5cb3a2c658.rna_seq.a...,TCGA-43-6773,TCGA-43-6773-11A,TCGA-43-6773-11A,Not Applicable,TS
504,TCGA-43-6143-01A-01-TS1.6716067a-f179-46cf-8d3...,3953b63a-1a63-4e97-b683-ae905b55e03e.rna_seq.a...,TCGA-43-6143,TCGA-43-6143-01A,TCGA-43-6143-01A,Primary,TS
505,TCGA-43-A56V-01A-01-TSA.6B4B05A2-0CAC-4779-9C8...,50f7bf08-3f8b-492b-afdc-40b4f35060bf.rna_seq.a...,TCGA-43-A56V,TCGA-43-A56V-01A,TCGA-43-A56V-01A,Primary,TS
506,TCGA-56-5897-01A-01-TS1.360c2b58-7187-4e91-a2e...,8175545d-b1a7-472e-aa7f-23adaece2ccd.rna_seq.a...,TCGA-56-5897,TCGA-56-5897-01A,TCGA-56-5897-01A,Primary,TS


In [12]:
metadata['id_pair'] = np.arange(len(metadata))

In [13]:
metadata.to_csv(f'data/metadata_{slide_type}.csv', index=False)